# 数据探索

本 Notebook 用于数据探索和分析。

In [7]:
import os
import pandas as pd
import json
from pathlib import Path
from glob import glob

# 设置路径
base_path = Path(r"D:\Jupyter profile\汽车信息安全风险评估")
raw_path = base_path / "data" / "raw"
processed_path = base_path / "data" / "processed"

# 确保输出目录存在
processed_path.mkdir(parents=True, exist_ok=True)

# 定义新的列名
new_columns = [
    '风险ID', '产品名称', '一级功能', '二级功能', '元素编号', '资产', '网络安全假设',
    '资产类别', '安全属性', '源名称', '目的名称', '损害场景', '安全分数', '安全说明',
    '财产分数', '财产说明', '操作分数', '操作说明', '隐私分数', '隐私说明',
    '影响级别参数', '影响评级', '威胁类型', 'R155对应项', '威胁场景描述', '攻击路径',
    '攻击向量', '已有安全措施', '暴露时间', '暴露说明', '专业经验', '专业说明',
    '所需信息', '信息说明', '机会窗口', '窗口说明', '所需设备', '设备说明',
    '攻击可行性参数和', '攻击可行性等级', '风险等级', 'CAL等级', '风险处置',
    '安全目标', '安全目标编号', '安全声明', '安全概念编号', '残余风险等级', '残余风险处置'
]


基础路径: D:\Jupyter profile\汽车信息安全风险评估
原始数据路径: D:\Jupyter profile\汽车信息安全风险评估\data\raw
处理后数据路径: D:\Jupyter profile\汽车信息安全风险评估\data\processed

目标列数: 49
列名预览: ['风险ID', '产品名称', '一级功能', '二级功能', '元素编号'] ... ['安全概念编号', '残余风险等级', '残余风险处置']


In [25]:
def process_excel_file(excel_path, new_columns):
    """
    处理单个Excel文件：读取最后一个sheet，去除前6行，重命名列
    """
    try:
        # 获取所有sheet名称
        xl = pd.ExcelFile(excel_path)
        sheet_names = xl.sheet_names
        
        if not sheet_names:
            print(f"  ⚠️ {excel_path.name} 没有sheet")
            return None
            
        # 读取最后一个sheet
        last_sheet = sheet_names[-1]
        print(f"  📄 读取Sheet: {last_sheet}")
        
        # 读取数据，跳过前6行（索引0-5），从第7行开始（索引6）
        df = pd.read_excel(excel_path, sheet_name=last_sheet, header=None, skiprows=6)
        
        # 如果列数不匹配，进行处理
        if len(df.columns) != len(new_columns):
            print(f"  ⚠️ 列数不匹配: 原始{len(df.columns)}列 vs 目标{len(new_columns)}列")
            
            # 如果原始列数多于目标，截取前N列
            if len(df.columns) > len(new_columns):
                df = df.iloc[:, :len(new_columns)]
                print(f"  ✂️ 截取前{len(new_columns)}列")
            else:
                # 如果原始列数少于目标，补充空列
                for i in range(len(new_columns) - len(df.columns)):
                    df[len(df.columns)] = None
                print(f"  ➕ 补充{len(new_columns) - len(df.columns)}个空列")
        
        # 重命名列
        df.columns = new_columns
        
        # 去除完全空白的行
        df = df.dropna(how='all')
        df = df.dropna(subset=['资产'])
        
        print(f"  ✅ 成功处理: {len(df)} 行数据")
        return df
        
    except Exception as e:
        print(f"  ❌ 处理失败: {str(e)}")
        return None

# 测试读取一个文件
test_dirs = [d for d in raw_path.iterdir() if d.is_dir()]
print(f"\n发现 {len(test_dirs)} 个数据目录:")
for d in test_dirs[:11]:
    print(f"  - {d.name}")

# 测试第一个Excel文件
if test_dirs:
    test_dir = test_dirs[8]
    excel_files = list(test_dir.glob("*.xlsx"))
    if excel_files:
        print(f"\n🧪 测试处理: {excel_files[0].name}")
        test_df = process_excel_file(excel_files[0], new_columns)
        if test_df is not None:
            print(f"\n数据预览（前3行）:")
            print(test_df.head(2).to_string())



发现 10 个数据目录:
  - G3-车联网
  - HW7
  - S20燃油
  - 优秀示例
  - 少林客车
  - 庆铃-轻卡4KB1
  - 无人车
  - 欧胜传统车
  - 欧胜新能源
  - 零跑（删RF ICU）

🧪 测试处理: 欧胜新能源TARA.xlsx
  📄 读取Sheet: 欧胜新能源TARA
  ✅ 成功处理: 4437 行数据

数据预览（前3行）:
                  风险ID   产品名称 一级功能  二级功能 元素编号   资产  网络安全假设 资产类别  安全属性  源名称 目的名称                                  损害场景  安全分数  安全说明  财产分数  财产说明  操作分数  操作说明  隐私分数  隐私说明  影响级别参数 影响评级  威胁类型                                                               R155对应项                                            威胁场景描述                                                                              攻击路径      攻击向量  已有安全措施  暴露时间  暴露说明  专业经验  专业说明  所需信息  信息说明  机会窗口  窗口说明  所需设备  设备说明  攻击可行性参数和 攻击可行性等级  风险等级 CAL等级  风险处置 安全目标 安全目标编号                                 安全声明  安全概念编号  残余风险等级  残余风险处置
0  Risk-欧胜新能源-新能源-4716  欧胜新能源  新能源  AVAS  BCM  BCM     NaN   部件  权限属性  BCM  BCM  资产BCM相关信息被篡改，权限属性被侵害，导致非法用户可篡改AVAS功能     1   NaN     1   NaN     1   NaN     2   NaN       2  中等的    越权                                             

In [ ]:
import os
import pandas as pd
import json
from pathlib import Path
from glob import glob
import re
from datetime import datetime

# ==================== 配置路径 ====================
base_path = Path(r"D:\Jupyter profile\汽车信息安全风险评估")
raw_path = base_path / "data" / "raw"
processed_path = base_path / "data" / "processed"

# 确保输出目录存在
processed_path.mkdir(parents=True, exist_ok=True)

# ==================== 定义列名 ====================
new_columns = [
    '风险ID', '产品名称', '一级功能', '二级功能', '元素编号', '资产', '网络安全假设',
    '资产类别', '安全属性', '源名称', '目的名称', '损害场景', '安全分数', '安全说明',
    '财产分数', '财产说明', '操作分数', '操作说明', '隐私分数', '隐私说明',
    '影响级别参数', '影响评级', '威胁类型', 'R155对应项', '威胁场景描述', '攻击路径',
    '攻击向量', '已有安全措施', '暴露时间', '暴露说明', '专业经验', '专业说明',
    '所需信息', '信息说明', '机会窗口', '窗口说明', '所需设备', '设备说明',
    '攻击可行性参数和', '攻击可行性等级', '风险等级', 'CAL等级', '风险处置',
    '安全目标', '安全目标编号', '安全声明', '安全概念编号', '残余风险等级', '残余风险处置'
]

# 基础输入字段（来自Excel）
base_input_fields = ['一级功能', '二级功能', '资产', '资产类别']

# 输出字段（用于构建Response）
output_fields = [
    '安全属性', '损害场景', '安全分数', '财产分数', '操作分数', '隐私分数',
    '威胁类型', '威胁场景描述', '攻击路径', '暴露时间', '专业经验', 
    '所需信息', '机会窗口', '所需设备'
]

print("✅ 配置完成")
print(f"📁 基础路径: {base_path}")
print(f"📊 目标列数: {len(new_columns)}")
print(f"📝 基础输入字段: {len(base_input_fields)} 个")
print(f"🎯 输出字段: {len(output_fields)} 个")

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置图表样式
sns.set_style('whitegrid')
plt.rcParams['font.sans-serif'] = ['SimHei']  # 支持中文
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

## 数据加载

在这里加载你的数据集。

In [ ]:
# 加载数据
# data = pd.read_csv(RAW_DATA_DIR / 'your_data.csv')
# print(f"数据形状: {data.shape}")
# data.head()

## 拓扑图

In [3]:
import json
import os
import glob

def generate_topology(raw_data):

    nodes_map = {}
    protocol_map = {}
    
    cells_list = []
    line_data_list = []
    
    # 1. 递归提取数据
    def extract_arrays(obj):
        if isinstance(obj, dict):
            if "cells" in obj and isinstance(obj["cells"], list):
                cells_list.extend(obj["cells"])
            if "lineData" in obj and isinstance(obj["lineData"], list):
                line_data_list.extend(obj["lineData"])
            for key, value in obj.items():
                if key not in ["cells", "lineData"]:
                    extract_arrays(value)
        elif isinstance(obj, list):
            for item in obj:
                extract_arrays(item)

    extract_arrays(raw_data)
    
    # 2. 建立协议/颜色映射表 (例如: #F53F3F -> CAN)
    for line in line_data_list:
        color = line.get("color", "").upper()
        name = line.get("name", "Unknown")
        if color:
            protocol_map[color] = name

    # 3. 提取所有节点的名字，建立 ID -> Name 映射
    for cell in cells_list:
        if cell.get("shape") in ["custom-rounded-rect", "custom-rounded-rect-dash", "custom-filled-rect"]:
            node_id = cell.get("id", "")
            text = cell.get("attrs", {}).get("text", {}).get("text", "未知节点")
            if node_id:
                nodes_map[node_id] = text

    # 4. 提取边
    unique_edges = {} # 使用字典去重
    
    for cell in cells_list:
        if cell.get("shape") == "edge":
            source_id = cell.get("source", {}).get("cell", "")
            target_id = cell.get("target", {}).get("cell", "")
            
            # 提取颜色并获取协议名
            stroke = cell.get("attrs", {}).get("line", {}).get("stroke", "#000000").upper()
            protocol_name = protocol_map.get(stroke, stroke)
            
            # 获取节点真实名称
            source_name = nodes_map.get(source_id, f"未知节点({source_id})")
            target_name = nodes_map.get(target_id, f"未知节点({target_id})")
            
            # 无向图去重逻辑
            # 将两个节点名称排序，这样 A->B 和 B->A 会生成相同的 key，防止重复添加
            sorted_nodes = tuple(sorted([source_name, target_name]))
            edge_key = (sorted_nodes[0], sorted_nodes[1], protocol_name, stroke)
            
            if edge_key not in unique_edges:
                unique_edges[edge_key] = {
                    "connected": [source_name, target_name], # 保持原顺序放入数组
                    "protocol": protocol_name,
                    "color": stroke
                }

    # 返回组装好的字典结构
    return {
        "topology": list(unique_edges.values())
    }

def batch_process_to_json(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    json_files = glob.glob(os.path.join(input_dir, "*.json"))
    
    if not json_files:
        print(f"未找到 JSON 文件。")
        return

    for file_path in json_files:
        filename = os.path.basename(file_path)
        # 保持 .json 后缀，因为输出目录不同，不会覆盖原文件
        output_path = os.path.join(output_dir, filename)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)
                
            # 直接获取方案 B 的字典数据
            topology_data = generate_topology(raw_data)
            
            if not topology_data.get("topology"):
                print(f"⚠️ 警告: {filename} 未提取到有效拓扑边。")
                continue
            
            # 直接将字典写入为 JSON 文件
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(topology_data, f, ensure_ascii=False, indent=2)
                
            print(f"✅ 成功转换为JSON: {filename}")
            
        except Exception as e:
            print(f"❌ 处理出错 {filename}: {e}")

if __name__ == "__main__":
    # 您的数据目录
    INPUT_FOLDER = r"D:\Jupyter profile\汽车信息安全风险评估\data\raw\拓扑图json"
    OUTPUT_FOLDER = r"D:\Jupyter profile\汽车信息安全风险评估\data\raw\拓扑图pre"
    
    batch_process_to_json(INPUT_FOLDER, OUTPUT_FOLDER)

⚠️ 警告: 11-ASE-TARA1.4.json 未提取到有效拓扑边。
⚠️ 警告: 15-ASE-TARA1.4.json 未提取到有效拓扑边。
⚠️ 警告: EHV-整车TARA报告V1.4.json 未提取到有效拓扑边。
⚠️ 警告: F2N.json 未提取到有效拓扑边。
⚠️ 警告: MY26.json 未提取到有效拓扑边。
⚠️ 警告: P20_TARA_v1.2-20260121.json 未提取到有效拓扑边。
⚠️ 警告: T9TARA -V1.2.json 未提取到有效拓扑边。
⚠️ 警告: 少林客车TARA-1.3.json 未提取到有效拓扑边。
⚠️ 警告: 庆铃-轻卡4KB1_TARA_v1.0--最终版.json 未提取到有效拓扑边。
⚠️ 警告: 智己S32L-TARA.json 未提取到有效拓扑边。
⚠️ 警告: 欧胜传统车TARA-V1.5.json 未提取到有效拓扑边。


In [11]:
import json
import os
import glob

def generate_topology_bus_centric(raw_data):
    """
    将原始 JSON 数据提取并转换为【按总线网段分组】的极简字典结构
    极大地节省 LLM 上下文 Token，并符合汽车总线物理特性
    """
    nodes_map = {}
    protocol_map = {}
    
    cells_list = []
    line_data_list = []
    
    # 1. 递归提取数据
    def extract_arrays(obj):
        if isinstance(obj, dict):
            if "cells" in obj and isinstance(obj["cells"], list):
                cells_list.extend(obj["cells"])
            if "lineData" in obj and isinstance(obj["lineData"], list):
                line_data_list.extend(obj["lineData"])
            for key, value in obj.items():
                if key not in ["cells", "lineData"]:
                    extract_arrays(value)
        elif isinstance(obj, list):
            for item in obj:
                extract_arrays(item)

    extract_arrays(raw_data)
    
    # 2. 建立协议/颜色映射表 (例如: #F53F3F -> CAN)
    for line in line_data_list:
        color = line.get("color", "").upper()
        name = line.get("name", "Unknown")
        if color:
            protocol_map[color] = name

    # 3. 提取所有节点的名字，建立 ID -> Name 映射
    for cell in cells_list:
        if cell.get("shape") in ["custom-rounded-rect", "custom-rounded-rect-dash", "custom-filled-rect"]:
            node_id = cell.get("id", "")
            text = cell.get("attrs", {}).get("text", {}).get("text", "未知节点")
            if node_id:
                nodes_map[node_id] = text

    # 4. 提取边，并按【总线/颜色】进行分组聚合
    bus_groups = {}
    
    for cell in cells_list:
        if cell.get("shape") == "edge":
            source_id = cell.get("source", {}).get("cell", "")
            target_id = cell.get("target", {}).get("cell", "")
            
            # 提取颜色并获取协议名
            stroke = cell.get("attrs", {}).get("line", {}).get("stroke", "#000000").upper()
            protocol_name = protocol_map.get(stroke, stroke)
            
            # 获取节点真实名称
            source_name = nodes_map.get(source_id, f"未知节点({source_id})")
            target_name = nodes_map.get(target_id, f"未知节点({target_id})")
            
            # 组合总线名称和颜色作为字典的 Key，例如 "BCAN(#016AFF)"
            bus_key = f"{protocol_name}({stroke})"
            
            # 如果这个总线还没记录过，初始化一个空集合（Set 用于自动去重）
            if bus_key not in bus_groups:
                bus_groups[bus_key] = set()
                
            # 将这条线两端的节点都挂载到这条总线上
            bus_groups[bus_key].add(source_name)
            bus_groups[bus_key].add(target_name)

    # 5. 将集合(Set)转换为列表(List)并排序，以便输出标准的 JSON
    # 最终格式: { "BCAN(#016AFF)": ["CGW", "BCM", "IVI", ...] }
    optimized_topology = {
        bus: sorted(list(nodes)) 
        for bus, nodes in bus_groups.items()
    }
    
    return optimized_topology

def batch_process_to_json(input_dir, output_dir):
    """批量将原始 JSON 转换为极简总线分组的 JSON 文件"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    json_files = glob.glob(os.path.join(input_dir, "*.json"))
    
    if not json_files:
        print(f"未找到 JSON 文件。")
        return

    for file_path in json_files:
        filename = os.path.basename(file_path)
        # 可以在文件名后加个标识，避免与原文件混淆，这里保持原名输出到新文件夹
        output_path = os.path.join(output_dir, filename)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)
                
            # 获取极简总线分组数据
            topology_data = generate_topology_bus_centric(raw_data)
            
            if not topology_data:
                print(f"⚠️ 警告: {filename} 未提取到有效拓扑数据。")
                continue
            
            # 写入 JSON 文件，ensure_ascii=False 保证中文正常显示
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(topology_data, f, ensure_ascii=False)
                
            print(f"✅ 成功转换为极简总线 JSON: {filename}")
            
        except Exception as e:
            print(f"❌ 处理出错 {filename}: {e}")

if __name__ == "__main__":
    # 修改为您的实际数据目录
    INPUT_FOLDER = r"C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\json\raw"
    OUTPUT_FOLDER = r"C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\json\pre"
    
    batch_process_to_json(INPUT_FOLDER, OUTPUT_FOLDER)

提取完成！共提取了 6 个零部件的接口信息，已保存至 C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\information\output.json


In [1]:
import json
import os
import glob

def generate_topology_bus_centric(raw_data):
    """
    将原始 JSON 数据提取并转换为【按总线网段分组】的结构化对象列表
    极大地节省 LLM 上下文 Token，并符合汽车总线物理特性
    """
    nodes_map = {}
    protocol_map = {}
    
    cells_list = []
    line_data_list = []
    
    # 1. 递归提取数据
    def extract_arrays(obj):
        if isinstance(obj, dict):
            if "cells" in obj and isinstance(obj["cells"], list):
                cells_list.extend(obj["cells"])
            if "lineData" in obj and isinstance(obj["lineData"], list):
                line_data_list.extend(obj["lineData"])
            for key, value in obj.items():
                if key not in ["cells", "lineData"]:
                    extract_arrays(value)
        elif isinstance(obj, list):
            for item in obj:
                extract_arrays(item)

    extract_arrays(raw_data)
    
    # 2. 建立协议/颜色映射表 (例如: #F53F3F -> CAN)
    for line in line_data_list:
        color = line.get("color", "").upper()
        name = line.get("name", "Unknown")
        if color:
            protocol_map[color] = name

    # 3. 提取所有节点的名字，建立 ID -> Name 映射
    for cell in cells_list:
        if cell.get("shape") in ["custom-rounded-rect", "custom-rounded-rect-dash", "custom-filled-rect"]:
            node_id = cell.get("id", "")
            text = cell.get("attrs", {}).get("text", {}).get("text", "未知节点")
            if node_id:
                nodes_map[node_id] = text

    # 4. 提取边，并按【总线颜色】进行分组聚合
    bus_groups = {}
    
    for cell in cells_list:
        if cell.get("shape") == "edge":
            source_id = cell.get("source", {}).get("cell", "")
            target_id = cell.get("target", {}).get("cell", "")
            
            # 提取颜色并获取协议名
            stroke = cell.get("attrs", {}).get("line", {}).get("stroke", "#000000").upper()
            protocol_name = protocol_map.get(stroke, stroke)
            
            # 获取节点真实名称
            source_name = nodes_map.get(source_id, f"未知节点({source_id})")
            target_name = nodes_map.get(target_id, f"未知节点({target_id})")
            
            # 以颜色作为唯一 Key 进行聚合，预设目标字典结构
            if stroke not in bus_groups:
                bus_groups[stroke] = {
                    "protocol": protocol_name,
                    "color": stroke,
                    "controller": set()  # 使用 Set 用于自动去重
                }
                
            # 将这条线两端的节点都挂载到这条总线的 controller 中
            bus_groups[stroke]["controller"].add(source_name)
            bus_groups[stroke]["controller"].add(target_name)

    # 5. 将集合(Set)转换为列表(List)并排序，组装成最终格式
    # 最终格式: [ { "protocol":"BCAN", "color":"#016AFF", "controller":["CGW", "BCM", "IVI"] }, ... ]
    optimized_topology = [
        {
            "protocol": data["protocol"],
            "color": data["color"],
            "controller": sorted(list(data["controller"]))
        }
        for data in bus_groups.values()
    ]
    
    return optimized_topology

def batch_process_to_json(input_dir, output_dir):
    """批量将原始 JSON 转换为极简总线分组的 JSON 文件"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    json_files = glob.glob(os.path.join(input_dir, "*.json"))
    
    if not json_files:
        print(f"未找到 JSON 文件。")
        return

    for file_path in json_files:
        filename = os.path.basename(file_path)
        output_path = os.path.join(output_dir, filename)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)
                
            topology_data = generate_topology_bus_centric(raw_data)
            
            if not topology_data:
                print(f"⚠️ 警告: {filename} 未提取到有效拓扑数据。")
                continue
            
            with open(output_path, 'w', encoding='utf-8') as f:
                # indent=4 可以让输出的 JSON 自动排版缩进，更易读（如果不需要可删除此参数以节省空间）
                json.dump(topology_data, f, ensure_ascii=False,indent=4 )
                
            print(f"✅ 成功转换为极简总线 JSON: {filename}")
            
        except Exception as e:
            print(f"❌ 处理出错 {filename}: {e}")

if __name__ == "__main__":
    INPUT_FOLDER = r"C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\json\raw"
    OUTPUT_FOLDER = r"C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\json\pre"
    
    batch_process_to_json(INPUT_FOLDER, OUTPUT_FOLDER)


✅ 成功转换为极简总线 JSON: 拓扑图数据导出2026_4_13 10_19_50.json


In [3]:
import json
import os
import glob

def generate_topology_bus_centric(raw_data):
    nodes_map = {}
    protocol_map = {}
    
    cells_list = []
    line_data_list = []
    
    # 1. 递归提取数据
    def extract_arrays(obj):
        if isinstance(obj, dict):
            if "cells" in obj and isinstance(obj["cells"], list):
                cells_list.extend(obj["cells"])
            if "lineData" in obj and isinstance(obj["lineData"], list):
                line_data_list.extend(obj["lineData"])
            for key, value in obj.items():
                if key not in ["cells", "lineData"]:
                    extract_arrays(value)
        elif isinstance(obj, list):
            for item in obj:
                extract_arrays(item)

    extract_arrays(raw_data)
    
    # 2. 建立协议/颜色映射表
    for line in line_data_list:
        color = line.get("color", "").upper()
        name = line.get("name", "Unknown")
        if color:
            protocol_map[color] = name

    # 3. 提取所有节点的名字，建立 ID -> Name 映射
    for cell in cells_list:
        if cell.get("shape") in ["custom-rounded-rect", "custom-rounded-rect-dash", "custom-filled-rect"]:
            node_id = cell.get("id", "")
            text = cell.get("attrs", {}).get("text", {}).get("text", "未知节点")
            if node_id:
                nodes_map[node_id] = text

    # 4. 提取边，并按【总线颜色】进行分组聚合
    bus_groups = {}
    
    for cell in cells_list:
        if cell.get("shape") == "edge":
            source_id = cell.get("source", {}).get("cell", "")
            target_id = cell.get("target", {}).get("cell", "")
            
            stroke = cell.get("attrs", {}).get("line", {}).get("stroke", "#000000").upper()
            protocol_name = protocol_map.get(stroke, stroke)
            
            source_name = nodes_map.get(source_id, f"未知节点({source_id})")
            target_name = nodes_map.get(target_id, f"未知节点({target_id})")
            
            if stroke not in bus_groups:
                bus_groups[stroke] = {
                    "protocol": protocol_name,
                    "controller": set()
                }
                
            bus_groups[stroke]["controller"].add(source_name)
            bus_groups[stroke]["controller"].add(target_name)

    # 5. 将集合转换为列表并排序，组装成最终格式
    # 直接将协议名作为键，节点列表作为值，并加上自增数字 id
    optimized_topology = [
        {
            "id": idx + 1,                       # 数字 id
            data["protocol"]: sorted(list(data["controller"])) # 动态键值对: "CAN": ["BCM", "CGW"]
        }
        for idx, data in enumerate(bus_groups.values())
    ]
    
    return optimized_topology

def batch_process_to_json(input_dir, output_dir):
    """批量将原始 JSON 转换为极简总线分组的 JSON 文件"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    json_files = glob.glob(os.path.join(input_dir, "*.json"))
    
    if not json_files:
        print(f"未找到 JSON 文件。")
        return

    for file_path in json_files:
        filename = os.path.basename(file_path)
        output_path = os.path.join(output_dir, filename)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)
                
            topology_data = generate_topology_bus_centric(raw_data)
            
            if not topology_data:
                print(f"⚠️ 警告: {filename} 未提取到有效拓扑数据。")
                continue
            
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(topology_data, f, ensure_ascii=False,indent=4 )
                
            print(f"✅ 成功转换为极简总线 JSON: {filename}")
            
        except Exception as e:
            print(f"❌ 处理出错 {filename}: {e}")

if __name__ == "__main__":
    INPUT_FOLDER = r"C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\json\raw"
    OUTPUT_FOLDER = r"C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\json\pre"
    
    batch_process_to_json(INPUT_FOLDER, OUTPUT_FOLDER)


✅ 成功转换为极简总线 JSON: 拓扑图数据导出2026_4_13 10_19_50.json
